In [1]:
from pathlib import Path
import json

# Our directories
sections_dir = Path("temporary_sections")
responses_dir = Path("../../data/test/raw_api_responses")

# Get all temporary audio sections
audio_files = sorted(sections_dir.glob("*.wav"))

# Get names of sections already transcribed
completed = {
    file.stem
    for file in responses_dir.glob("section_*.response")
}

# Find sections that still need transcription
pending = [
    file for file in audio_files
    if file.stem not in completed
]

print("Total audio sections:", len(audio_files))
print("Already transcribed:", len(audio_files) - len(pending))
print("Still need transcription:", len(pending))

print("\nFirst 5 pending sections:")
for file in pending[:5]:
    print(file.name)

Total audio sections: 10
Already transcribed: 10
Still need transcription: 0

First 5 pending sections:


In [2]:
import subprocess

original_audio = Path("../../data/test/test_audio.mp3")

def get_duration(path):
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path)
        ],
        capture_output=True,
        text=True,
        check=True
    )

    return float(result.stdout.strip())

if original_audio.exists():
    duration = get_duration(original_audio)

    print("Original duration:", round(duration, 2), "seconds")
    print("Original duration:", round(duration / 60, 2), "minutes")
else:
    print("Original audio not found:", original_audio)

Original duration: 591.73 seconds
Original duration: 9.86 minutes


In [3]:
metadata_path = Path("temporary_sections.json")

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

print("Metadata type:", type(metadata).__name__)

if isinstance(metadata, list):
    print("Number of entries:", len(metadata))
    print(json.dumps(metadata[:2], indent=2, ensure_ascii=False))

elif isinstance(metadata, dict):
    print("Top-level keys:", list(metadata.keys()))
    print(
        json.dumps(metadata, indent=2, ensure_ascii=False)[:2000]
    )

Metadata type: list
Number of entries: 10
[
  {
    "filename": "section_000.wav",
    "start": 0,
    "end": 59.9
  },
  {
    "filename": "section_001.wav",
    "start": 58.9,
    "end": 122.55
  }
]


In [4]:
import re
from rapidfuzz import fuzz

# Read the entire book
book = Path("../../data/test/book.txt").read_text(
    encoding="utf-8"
)

# Normalize text for matching
def normalize(text):
    text = text.lower()

    for mark in ["‘", "’", "ʻ", "ʼ", "`"]:
        text = text.replace(mark, "'")

    text = re.sub(r"[^\w\s']", " ", text)
    return " ".join(text.split())


book_words = normalize(book).split()

# Read all existing transcription responses
response_files = sorted(
    responses_dir.glob("section_*.response")
)

matches = []
previous_start = 0
previous_end = 0

for index, response_path in enumerate(response_files):

    data = json.loads(
        response_path.read_text(encoding="utf-8")
    )

    transcription = " ".join(
        segment["text"].strip()
        for segment in data["segments"]
    )

    spoken_text = normalize(transcription)
    word_count = len(spoken_text.split())

    if word_count == 0:
        print("Empty transcription:", response_path.name)
        continue

    # Allow different book passage lengths
    window_sizes = [
        max(1, int(word_count * factor))
        for factor in [0.8, 1.0, 1.2]
    ]

    # First section: search the entire book
    if index == 0:
        search_start = 0
        search_end = len(book_words) - 1

    # Other sections: search near previous match
    else:
        search_start = max(0, previous_start - 30)
        search_end = min(
            len(book_words) - 1,
            previous_end + word_count * 3
        )

    best_score = -1
    best_start = 0
    best_end = 0

    # Search in small steps
    step = max(1, word_count // 10)

    for start in range(search_start, search_end + 1, step):

        for size in window_sizes:

            end = min(start + size, len(book_words))

            candidate = " ".join(book_words[start:end])

            score = fuzz.ratio(spoken_text, candidate)

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    # Refine the best position word by word
    for start in range(
        max(search_start, best_start - step),
        min(search_end, best_start + step) + 1
    ):
        for size in window_sizes:

            end = min(start + size, len(book_words))
            candidate = " ".join(book_words[start:end])
            score = fuzz.ratio(spoken_text, candidate)

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    previous_start = best_start
    previous_end = best_end

    matched_text = " ".join(
        book_words[best_start:best_end]
    )

    matches.append({
        "section": response_path.stem,
        "start_word": best_start,
        "end_word": best_end,
        "similarity": round(best_score, 2),
        "transcription": transcription,
        "book_text": matched_text,
        "needs_review": best_score < 85
    })

    print(
        response_path.stem,
        "| Score:", round(best_score, 2),
        "| Words:", best_start, "-", best_end
    )

# Save matching results
output_path = Path("../../data/test/book_matches_draft.json")

output_path.write_text(
    json.dumps(matches, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nSaved:", output_path)

section_000 | Score: 82.35 | Words: 0 - 51
section_001 | Score: 98.65 | Words: 51 - 151
section_002 | Score: 95.03 | Words: 150 - 237
section_003 | Score: 98.26 | Words: 240 - 330
section_004 | Score: 96.54 | Words: 331 - 452
section_005 | Score: 98.93 | Words: 453 - 550
section_006 | Score: 97.39 | Words: 550 - 642
section_007 | Score: 98.91 | Words: 640 - 738
section_008 | Score: 93.84 | Words: 738 - 823
section_009 | Score: 94.55 | Words: 828 - 890

Saved: ../../data/test/book_matches_draft.json
